# Two-Body Weber Electrodynamics (Cartesian vs Adaptive vs Lifted Pair)


In [ ]:
using WeberElectrodynamics
using LinearAlgebra
using Plots
using Printf


## 1. System Construction


In [ ]:
# Physical parameters
m1, m2 = 1.0, 0.1
q1, q2 = sqrt(0.1), -sqrt(0.1)  # attractive
c = 4.0
k = q1 * q2

system = WeberSystem(2, 2)

println("Particles: ", system.n_particles)
println("Dimensions: ", system.dims)
println("Degrees of freedom: ", system.degrees_of_freedom)


## 2. Symbolic Hamiltonian


In [ ]:
system.hamiltonian_symbolic


## 3. Near-Collision Initial Condition


In [ ]:
r0 = 2.0
M = m1 + m2
mu = m1 * m2 / M
v_circ = sqrt(abs(k) / (mu * r0))
v_scale = 0.2  # close-encounter regime

q0 = [-m2 / M * r0, 0.0, m1 / M * r0, 0.0]
p0 = [0.0, m1 * (-m2 / M * v_circ * v_scale),
      0.0, m2 * (m1 / M * v_circ * v_scale)]

tspan = (0.0, 3.0)
dt = 0.004
dt_ref = dt / 4

println("v_scale = ", v_scale)
println("tspan   = ", tspan)
println("dt      = ", dt)
println("dt_ref  = ", dt_ref)


## 4. Problem Setup (A/B/C + Fine Reference)


In [ ]:
prob_A = WeberProblem(system, tspan, q0, p0;
    masses=[m1, m2], charges=[q1, q2], c=c, dt=dt,
    regularization_enabled=false)

prob_B = WeberProblem(system, tspan, q0, p0;
    masses=[m1, m2], charges=[q1, q2], c=c, dt=dt,
    regularization_enabled=true,
    regularization_backend=:adaptive_cartesian,
    regularization_r_on=0.6,
    regularization_r_off=0.9,
    regularization_max_substeps=256)

prob_C = WeberProblem(system, tspan, q0, p0;
    masses=[m1, m2], charges=[q1, q2], c=c, dt=dt,
    regularization_enabled=true,
    regularization_backend=:lifted_pair,
    regularization_r_on=0.6,
    regularization_r_off=0.9,
    regularization_max_substeps=256)

prob_ref = WeberProblem(system, tspan, q0, p0;
    masses=[m1, m2], charges=[q1, q2], c=c, dt=dt_ref,
    regularization_enabled=true,
    regularization_backend=:lifted_pair,
    regularization_r_on=0.6,
    regularization_r_off=0.9,
    regularization_max_substeps=256)


## 5. Integration


In [ ]:
sol_A = solve(prob_A)
sol_B = solve(prob_B)
sol_C = solve(prob_C)
sol_ref = solve(prob_ref)

println("Run A retcode: ", sol_A.retcode)
println("Run B retcode: ", sol_B.retcode)
println("Run C retcode: ", sol_C.retcode)
println("Reference retcode: ", sol_ref.retcode)

println("Run B pair/adaptive steps: ", sol_B.regularization.pair_steps, " / ", sol_B.regularization.adaptive_pair_steps)
println("Run C pair/lifted steps: ", sol_C.regularization.pair_steps, " / ", sol_C.regularization.lifted_pair_steps)
println("Run C max substeps: ", sol_C.regularization.max_substeps_used)


## 6. Trajectory Analysis


In [ ]:
traj_A = compute_trajectory_data(sol_A, 2, 2; stride=1)
traj_B = compute_trajectory_data(sol_B, 2, 2; stride=1)
traj_C = compute_trajectory_data(sol_C, 2, 2; stride=1)

plot(
    plot_trajectories(traj_A),
    plot_trajectories(traj_B),
    plot_trajectories(traj_C),
    layout=(1,3),
    size=(2400,800),
    plot_title="Trajectories: A Cartesian vs B Adaptive vs C Lifted",
)


## 7. Energy Conservation


In [ ]:
energy_A = compute_energy_timeseries(sol_A; stride=1)
energy_B = compute_energy_timeseries(sol_B; stride=1)
energy_C = compute_energy_timeseries(sol_C; stride=1)

println("Run A max global error (%): ", @sprintf("%.4e", energy_A.statistics.global_error_percent_max))
println("Run B max global error (%): ", @sprintf("%.4e", energy_B.statistics.global_error_percent_max))
println("Run C max global error (%): ", @sprintf("%.4e", energy_C.statistics.global_error_percent_max))


In [ ]:
plot(
    plot_energy(energy_A),
    plot_energy(energy_B),
    plot_energy(energy_C),
    layout=(1,3),
    size=(2400,900),
    plot_title="Energy: A Cartesian vs B Adaptive vs C Lifted",
)


In [ ]:
plot(
    plot_pair_energy(energy_A, (1, 2)),
    plot_pair_energy(energy_B, (1, 2)),
    plot_pair_energy(energy_C, (1, 2)),
    layout=(1,3),
    size=(2400,900),
    plot_title="Pair Energy: A Cartesian vs B Adaptive vs C Lifted",
)


In [ ]:
plot(
    plot_energy_errors(energy_A),
    plot_energy_errors(energy_B),
    plot_energy_errors(energy_C),
    layout=(1,3),
    size=(2400,1200),
    plot_title="Energy Errors: A Cartesian vs B Adaptive vs C Lifted",
)


## 8. Force Analysis


In [ ]:
forces_A = compute_pair_force_timeseries(sol_A, (1, 2), 2, 2, [m1, m2], [q1, q2], c; stride=1)
forces_B = compute_pair_force_timeseries(sol_B, (1, 2), 2, 2, [m1, m2], [q1, q2], c; stride=1)
forces_C = compute_pair_force_timeseries(sol_C, (1, 2), 2, 2, [m1, m2], [q1, q2], c; stride=1)

println("Run A force mean: ", @sprintf("%.4e", forces_A.stats.mean))
println("Run B force mean: ", @sprintf("%.4e", forces_B.stats.mean))
println("Run C force mean: ", @sprintf("%.4e", forces_C.stats.mean))


In [ ]:
plot(
    plot_pair_forces(forces_A),
    plot_pair_forces(forces_B),
    plot_pair_forces(forces_C),
    layout=(1,3),
    size=(2400,1800),
    plot_title="Pair Forces: A Cartesian vs B Adaptive vs C Lifted",
)


## 9. Phase Space


In [ ]:
plot(
    plot_phase_space(forces_A),
    plot_phase_space(forces_B),
    plot_phase_space(forces_C),
    layout=(1,3),
    size=(2400,800),
    plot_title="Phase Space: A Cartesian vs B Adaptive vs C Lifted",
)


## 10. Momentum Conservation


In [ ]:
momentum_A = compute_momentum_timeseries(sol_A; stride=1)
momentum_B = compute_momentum_timeseries(sol_B; stride=1)
momentum_C = compute_momentum_timeseries(sol_C; stride=1)

plot(
    plot_momentum(momentum_A),
    plot_momentum(momentum_B),
    plot_momentum(momentum_C),
    layout=(1,3),
    size=(2400,900),
    plot_title="Momentum: A Cartesian vs B Adaptive vs C Lifted",
)


## 11. Comparison Summary


In [ ]:
state_error(sol, ref) = norm(sol.q[end] - ref.q[end]) + norm(sol.p[end] - ref.p[end])

min_sep_A = minimum(forces_A.phase_space.separation_distance)
min_sep_B = minimum(forces_B.phase_space.separation_distance)
min_sep_C = minimum(forces_C.phase_space.separation_distance)

println("Minimum separation:")
println("  Run A: ", @sprintf("%.6f", min_sep_A))
println("  Run B: ", @sprintf("%.6f", min_sep_B))
println("  Run C: ", @sprintf("%.6f", min_sep_C))

println("\nEnergy drift (% max global):")
println("  Run A: ", @sprintf("%.6e", energy_A.statistics.global_error_percent_max))
println("  Run B: ", @sprintf("%.6e", energy_B.statistics.global_error_percent_max))
println("  Run C: ", @sprintf("%.6e", energy_C.statistics.global_error_percent_max))

println("\nFinal-state error vs fine reference:")
println("  Run A: ", @sprintf("%.6e", state_error(sol_A, sol_ref)))
println("  Run B: ", @sprintf("%.6e", state_error(sol_B, sol_ref)))
println("  Run C: ", @sprintf("%.6e", state_error(sol_C, sol_ref)))

println("\nDiagnostics (Run B adaptive):")
println("  used backend: ", sol_B.regularization.used_backend)
println("  pair/adaptive/lifted: ", sol_B.regularization.pair_steps, " / ", sol_B.regularization.adaptive_pair_steps, " / ", sol_B.regularization.lifted_pair_steps)
println("  fallback steps: ", sol_B.regularization.backend_fallback_steps)
println("  max substeps used: ", sol_B.regularization.max_substeps_used)

println("\nDiagnostics (Run C lifted):")
println("  used backend: ", sol_C.regularization.used_backend)
println("  pair/adaptive/lifted: ", sol_C.regularization.pair_steps, " / ", sol_C.regularization.adaptive_pair_steps, " / ", sol_C.regularization.lifted_pair_steps)
println("  fallback steps: ", sol_C.regularization.backend_fallback_steps)
println("  max substeps used: ", sol_C.regularization.max_substeps_used)


## API Summary

- A (Cartesian): `regularization_enabled=false`
- B (Adaptive pair): `regularization_enabled=true, regularization_backend=:adaptive_cartesian`
- C (Lifted pair): `regularization_enabled=true, regularization_backend=:lifted_pair`
- Fallback rule: requesting `:lifted_pair` in 1D/3D falls back to `:adaptive_cartesian`
